# Pipeline complet sur Kaggle (GPU T4/P100, Internet activé)
Recrée le jeu de données depuis Wikimedia Commons (même sélection que `data/sources.csv`), puis baseline → LoRA rangs 4/8/16/32 → évaluation.
Tout le résultat est zippé dans `/kaggle/working/lora_zellige_results.zip` (onglet *Output*).
Conseil : **Save Version → Save & Run All** pour que ça tourne en arrière-plan (≈ 2 h).

In [ ]:
import os, sys, subprocess, time
T0 = time.time()
os.chdir('/kaggle/working')
if not os.path.exists('lora-zellige'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/braziyass/lora-zellige.git'], check=True)
os.chdir('/kaggle/working/lora-zellige')
subprocess.run(['git', 'pull', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
import torch; print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN')

In [ ]:
# Données : téléchargement (traçabilité dans un CSV à part) puis préparation avec les légendes du dépôt
for cat, n in [('Zellij in Morocco', 80), ('Zellij fountains in Morocco', 30)]:
    r = subprocess.run([sys.executable, 'data/download_commons.py', '--category', cat, '--limit', str(n),
                        '--sources', '/kaggle/working/sources_downloaded.csv'], capture_output=True, text=True)
    print(r.stdout[-400:], r.stderr[-400:])
!python -m src.prepare_data --config configs/base.yaml

In [ ]:
# Baseline
!python -m src.generate --config configs/base.yaml --base_only

In [ ]:
# Balayage des rangs 4/8/16/32 (checkpoints 250/500/1000/1500)
!python -m src.sweep --sweep configs/sweep.yaml

In [ ]:
# Génération de tous les checkpoints + évaluation
!python -m src.generate --config configs/base.yaml
!python -m src.evaluate --config configs/base.yaml

In [ ]:
# Archive des résultats (CSV, figures, LoRA, journaux d'entraînement, échantillon d'images)
import shutil, glob, json
os.makedirs('/kaggle/working/export', exist_ok=True)
for p in glob.glob('results/*.csv') + glob.glob('results/*.json'): shutil.copy(p, '/kaggle/working/export/')
shutil.copytree('results/figures', '/kaggle/working/export/figures', dirs_exist_ok=True)
shutil.copytree('outputs/loras', '/kaggle/working/export/loras', dirs_exist_ok=True)
shutil.copy('data/processed/report.json', '/kaggle/working/export/data_report.json')
shutil.make_archive('/kaggle/working/lora_zellige_results', 'zip', '/kaggle/working/export')
print('Durée totale : %.1f min' % ((time.time() - T0) / 60))
print(open('results/metrics_summary.csv').read()[:6000])